# Neural Machine Translation Assignment
In this notebook `myG2P`: https://github.com/ye-kyaw-thu/myG2P dataset was used.
Sequence-to-Sequence NMT models and Transformer Model are experience in this assignment.

## Enviroment Preparation
We will require [marian](https://github.com/marian-nmt/marian) and [mosesdecoder](https://github.com/moses-smt/mosesdecoder.git) for this assignment.

In [2]:
# 1. Install necessary build-essential libraries
!apt-get update && apt-get install -y libboost-system-dev libgoogle-perftools-dev

# 2. Clone the official codebase
!git clone https://github.com/marian-nmt/marian

# 3. Compile using CMake (It will automatically detect Kaggle's CUDA toolkit)
!mkdir -p marian/build
%cd marian/build
!cmake .. -DCOMPILE_CPU=on
!make -j$(nproc)

# 4. Jump back to your primary working directory
%cd /kaggle/working

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,303 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]          
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InReleas

In [ ]:
%%bash
# 1. Update package lists and install required build systems
sudo apt-get update -y
sudo apt-get install -y g++ git automake libtool zlib1g-dev libboost-all-dev libbz2-dev liblzma-dev

# 2. Navigate to working space and clone repository
cd /kaggle/working
git clone https://github.com/moses-smt/mosesdecoder.git mosesdecoder
cd mosesdecoder

# 3. Compile Moses via Boost Jam engine (utilizing 4 threads for speed)
./bjam -j4

In [13]:
!marian/build/marian -h

Marian: Fast Neural Machine Translation in C++
Usage: marian/build/marian [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=2048               Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), critical, off
  --log-tim

In [22]:
!mosesdecoder/bin/moses2 -h

Starting...
Moses - A beam search decoder for phrase-based statistical machine translation models
Copyright (C) 2006 University of Edinburgh

This library is free software; you can redistribute it and/or
modify it under the terms of the GNU Lesser General Public
License as published by the Free Software Foundation; either
version 2.1 of the License, or (at your option) any later version.

This library is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU
Lesser General Public License for more details.

You should have received a copy of the GNU Lesser General Public
License along with this library; if not, write to the Free Software
Foundation, Inc., 51 Franklin Street, Fifth Floor, Boston, MA  02110-1301  USA

***********************************************************************

Built on Jun 14 2026 at 09:48:06

WHO'S FAULT IS THIS GODDAM SOFTWARE:
Chris Dye

## Dataset Preparation

In [3]:
%%bash
rm -rf .git
git init -q

git remote add -f origin https://github.com/thonenyangal/AIE-F.git

git config core.sparseCheckout true
echo "assignment-submission/class-13and14/assignment-6_aung_hein/*" >> .git/info/sparse-checkout

git pull origin main

mv assignment-submission/class-13and14/assignment-6_aung_hein/g2p-par ./

rm -rf assignment-submission
rm -rf .git

Updating origin


From https://github.com/thonenyangal/AIE-F
 * [new branch]        main       -> origin/main
From https://github.com/thonenyangal/AIE-F
 * branch              main       -> FETCH_HEAD
Updating files: 100% (9646/9646), done.


In [6]:
!ls -la g2p-par

total 2108
drwxr-xr-x 3 root root   4096 Jun 14 09:27 .
drwxr-xr-x 5 root root   4096 Jun 14 09:27 ..
-rw-r--r-- 1 root root  59210 Jun 14 09:27 dev_clean.my
-rw-r--r-- 1 root root  25849 Jun 14 09:27 dev_clean.ph
-rw-r--r-- 1 root root  59222 Jun 14 09:27 dev.my
-rw-r--r-- 1 root root  25849 Jun 14 09:27 dev.ph
drwxr-xr-x 2 root root   4096 Jun 14 09:27 sgm
-rw-r--r-- 1 root root  83945 Jun 14 09:27 test_clean.my
-rw-r--r-- 1 root root  36532 Jun 14 09:27 test_clean.ph
-rw-r--r-- 1 root root  83959 Jun 14 09:27 test.my
-rw-r--r-- 1 root root  36532 Jun 14 09:27 test.ph
-rw-r--r-- 1 root root 594174 Jun 14 09:27 train_clean.my
-rw-r--r-- 1 root root 260356 Jun 14 09:27 train_clean.ph
-rw-r--r-- 1 root root 594183 Jun 14 09:27 train.my
-rw-r--r-- 1 root root 260356 Jun 14 09:27 train.ph


In [23]:
from pathlib import Path
import subprocess, re, shutil, os, difflib

BASE_DIR = Path('/kaggle/working')
G2P_DIR = BASE_DIR / 'g2p-par'
SGM_DIR = G2P_DIR / 'sgm'
MOSES_SRC_DIR = Path('/kaggle/working/mosesdecoder')
MOSES_SCRIPT_DIR = MOSES_SRC_DIR / 'scripts'
MOSES_BIN_DIR = MOSES_SRC_DIR / 'bin'
EXPERIMENT_PERL = MOSES_SCRIPT_DIR / 'ems' / 'experiment.perl'
MTEVAL = MOSES_SCRIPT_DIR / 'generic' / 'mteval-v13a.pl'

# The original assignment experiment directory.
ORIGINAL_EXP_DIR = BASE_DIR / 'pbsmt-big-normalize'
ORIGINAL_TEMPLATE = ORIGINAL_EXP_DIR / 'config.baseline'
ORIGINAL_BASELINE_DIR = ORIGINAL_EXP_DIR / 'baseline'

# New clean experiment directory.
EXP_DIR = BASE_DIR / 'pbsmt_original_clean_experiments'

print('BASE_DIR:', BASE_DIR)
print('G2P_DIR:', G2P_DIR)
print('SGM_DIR:', SGM_DIR)
print('Original template:', ORIGINAL_TEMPLATE, ORIGINAL_TEMPLATE.exists())
print('Clean experiment dir:', EXP_DIR)

BASE_DIR: /kaggle/working
G2P_DIR: /kaggle/working/g2p-par
SGM_DIR: /kaggle/working/g2p-par/sgm
Original template: /kaggle/working/pbsmt-big-normalize/config.baseline False
Clean experiment dir: /kaggle/working/pbsmt_original_clean_experiments


In [24]:
required = [
    G2P_DIR/'train_clean.my', G2P_DIR/'train_clean.ph',
    G2P_DIR/'dev_clean.my', G2P_DIR/'dev_clean.ph',
    G2P_DIR/'test_clean.my', G2P_DIR/'test_clean.ph',
]
for p in required:
    print(p, 'exists=', p.exists())
    if p.exists():
        print('  lines:', sum(1 for _ in p.open(encoding='utf-8')))
assert all(p.exists() for p in required), 'Missing cleaned train/dev/test files.'

/kaggle/working/g2p-par/train_clean.my exists= True
  lines: 20000
/kaggle/working/g2p-par/train_clean.ph exists= True
  lines: 20000
/kaggle/working/g2p-par/dev_clean.my exists= True
  lines: 2000
/kaggle/working/g2p-par/dev_clean.ph exists= True
  lines: 2000
/kaggle/working/g2p-par/test_clean.my exists= True
  lines: 2802
/kaggle/working/g2p-par/test_clean.ph exists= True
  lines: 2802


### Vocab
Let build the Vocab files.

In [25]:
!mkdir -p vocab

In [26]:
!mkdir -p preprocessing
!cat {G2P_DIR}/'train_clean.my' {G2P_DIR}/'dev_clean.my' > ./preprocessing/train-dev.my
!cat {G2P_DIR}/'train_clean.ph' {G2P_DIR}/'dev_clean.ph' > ./preprocessing/train-dev.ph

In [27]:
!wc -l ./preprocessing/*.{my,ph}

 22000 ./preprocessing/train-dev.my
 22000 ./preprocessing/train-dev.ph
 44000 total


In [28]:
!marian/build/marian-vocab < ./preprocessing/train-dev.my > ./vocab/vocab.my.yml

[2026-06-14 09:55:00] Creating vocabulary...
[2026-06-14 09:55:00] [data] Creating vocabulary stdout from stdin
[2026-06-14 09:55:01] Finished


In [29]:
!wc -l ./vocab/vocab.my.yml

2299 ./vocab/vocab.my.yml


In [30]:
!head ./vocab/vocab.my.yml

</s>: 0
<unk>: 1
အ: 2
မ: 3
သ: 4
က: 5
တ: 6
လက်: 7
ပ: 8
စ: 9


In [31]:
!marian/build/marian-vocab < ./preprocessing/train-dev.ph > ./vocab/vocab.ph.yml

[2026-06-14 09:55:21] Creating vocabulary...
[2026-06-14 09:55:21] [data] Creating vocabulary stdout from stdin
[2026-06-14 09:55:21] Finished


In [32]:
!wc -l ./vocab/vocab.ph.yml

1850 ./vocab/vocab.ph.yml


In [33]:
!head ./vocab/vocab.ph.yml

</s>: 0
<unk>: 1
a-: 2
ma-: 3
da-: 4
le': 5
jei: 6
ga-: 7
ta-: 8
tha-: 9


### Grapheme-to-Phoneme Translation with Sequence-to-Sequence Model

In [44]:
%%writefile /kaggle/working/seq2seq.myph.sh
#!/bin/bash

# ==============================================================================
# ENVIRONMENT PATHS
# ==============================================================================
BASE_DIR="/kaggle/working"
MODEL_FOLDER="${BASE_DIR}/model1.seq2seq.myph"
mkdir -p ${MODEL_FOLDER}

DATA_PATH="${BASE_DIR}/g2p-par" 
MARIAN_BIN="${BASE_DIR}/marian/build/marian"

src="my"
tgt="ph"

rm -f ${MODEL_FOLDER}/model.npz

# ==============================================================================
# MARIAN CONFIGURATION (HARDWARE-TUNED FOR TESLA T4 GPU)
# ==============================================================================
${MARIAN_BIN} \
  --type s2s \
  --train-sets ${DATA_PATH}/train_clean.${src} ${DATA_PATH}/train_clean.${tgt} \
  --max-length 100 \
  --valid-sets ${DATA_PATH}/dev_clean.${src} ${DATA_PATH}/dev_clean.${tgt} \
  --vocabs ${BASE_DIR}/vocab/vocab.${src}.yml ${BASE_DIR}/vocab/vocab.${tgt}.yml \
  --model ${MODEL_FOLDER}/model.npz \
  --devices 0 \
  --workspace 9000 \
  --mini-batch-fit \
  --enc-depth 2 --enc-type alternating --enc-cell lstm --enc-cell-depth 2 \
  --dec-depth 2 --dec-cell lstm --dec-cell-base-depth 2 --dec-cell-high-depth 2 \
  --tied-embeddings --layer-normalization --skip \
  --learn-rate 0.0001 \
  --clip-norm 1.0 \
  --valid-mini-batch 32 \
  --valid-metrics cross-entropy perplexity bleu \
  --valid-freq 1000 --save-freq 2000 --disp-freq 50 \
  --dropout-rnn 0.1 --dropout-src 0.1 --exponential-smoothing \
  --early-stopping 10 \
  --log ${MODEL_FOLDER}/train.log --valid-log ${MODEL_FOLDER}/valid.log \
  --seed 1111 \
  --dump-config > ${MODEL_FOLDER}/config.yml

echo "Launching high-throughput GPU training process..."
time ${MARIAN_BIN} -c ${MODEL_FOLDER}/config.yml 2>&1 | tee ${MODEL_FOLDER}/s2s.${src}-${tgt}.log

Overwriting /kaggle/working/seq2seq.myph.sh


#### Sequence to Sequence NMT training

In [45]:
!chmod +x seq2seq.myph.sh
!./seq2seq.myph.sh

Launching high-throughput GPU training process...
[2026-06-14 11:13:28] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-06-14 11:13:28] [marian] Running on f5f7ac2aebe0 as process 24107 with command line:
[2026-06-14 11:13:28] [marian] /kaggle/working/marian/build/marian -c /kaggle/working/model1.seq2seq.myph/config.yml
[2026-06-14 11:13:28] [config] after: 0e
[2026-06-14 11:13:28] [config] after-batches: 0
[2026-06-14 11:13:28] [config] after-epochs: 0
[2026-06-14 11:13:28] [config] all-caps-every: 0
[2026-06-14 11:13:28] [config] allow-unk: false
[2026-06-14 11:13:28] [config] authors: false
[2026-06-14 11:13:28] [config] beam-size: 12
[2026-06-14 11:13:28] [config] bert-class-symbol: "[CLS]"
[2026-06-14 11:13:28] [config] bert-mask-symbol: "[MASK]"
[2026-06-14 11:13:28] [config] bert-masking-fraction: 0.15
[2026-06-14 11:13:28] [config] bert-sep-symbol: "[SEP]"
[2026-06-14 11:13:28] [config] bert-train-type-embeddings: true
[2026-06-14 11:13:28] [config] bert-type-v

In [53]:
!time marian/build/marian-decoder -m model1.seq2seq.myph/model.npz -v vocab/vocab.my.yml vocab/vocab.ph.yml --devices 0 < g2p-par/test_clean.my > seq2seq.myph.hyp.txt

[2026-06-14 13:37:51] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-06-14 13:37:51] [marian] Running on f5f7ac2aebe0 as process 24170 with command line:
[2026-06-14 13:37:51] [marian] marian/build/marian-decoder -m model1.seq2seq.myph/model.npz -v vocab/vocab.my.yml vocab/vocab.ph.yml --devices 0
[2026-06-14 13:37:51] [config] alignment: ""
[2026-06-14 13:37:51] [config] allow-special: false
[2026-06-14 13:37:51] [config] allow-unk: false
[2026-06-14 13:37:51] [config] authors: false
[2026-06-14 13:37:51] [config] beam-size: 12
[2026-06-14 13:37:51] [config] bert-class-symbol: "[CLS]"
[2026-06-14 13:37:51] [config] bert-mask-symbol: "[MASK]"
[2026-06-14 13:37:51] [config] bert-masking-fraction: 0.15
[2026-06-14 13:37:51] [config] bert-sep-symbol: "[SEP]"
[2026-06-14 13:37:51] [config] bert-train-type-embeddings: true
[2026-06-14 13:37:51] [config] bert-type-vocab-size: 2
[2026-06-14 13:37:51] [config] best-deep: false
[2026-06-14 13:37:51] [config] build-info: ""
[20

In [54]:
!paste g2p-par/test_clean.my seq2seq.myph.hyp.txt | head -n 30 


တက် တက် ပြောင်	te' te' pjaun
ကပ် ပိ	ka' pi.
ရှုံ့ မဲ့	shoun. me.
ညှဉ်း ပန်း	njhin: ban:
မွမ်း မံ	mun: man
ငယ် မည်	nge mji
ရှာ ရှာ ဖွေ ဖွေ	sha sha hpwei hpwei
ပါး ပျဉ်း	pa- pjin:
ဥ မ ကွဲ သိုက် မ ပျက်	u. ma- gwe: thai' ma- pje'
အ စစ်	a- si'
ရေ ကျ	jei gya.
မ ဆုတ် မ ဆိုင်း	ma- hsou' ma- hsain:
ဟာ ကွက်	ha gwe'
မိန်း မူး	mein: mu:
ကယ် မ	ke ma.
နင်း နယ်	nin: ne
က ကြိုး တန် ဆာ	ka. gyou: tan za
ဆွမ်း ကြီး လောင်း	hswan: gyi: laun:
ဝါ ကြင့် ကြင့်	wa kyin. gyin.
ကုတ် ဟီး နာ	kou' hi: na
နီ ကြင် ကြင်	ni kyin gyin
အ လို တူ	a- lou du
ကိန်း ရင်း	kein: jin:
မီး တိုင်	mi: dain
ဝေ့ လည် ကြောင် ပတ်	wei. le kyaun ba'
စ ကား ပြော ကြေး နန်း	za- ga- pjo: kyi: nan:
အ ချစ် ဦး	a- chi' u:
အ သည်း ကောင်း	a- the: kaun:
ည ကြီး	nja. gyi:
ချို မြ မြ	chou mja. mja.
paste: write error: Broken pipe
paste: write error


In [60]:
!perl mosesdecoder/scripts/generic/multi-bleu.perl g2p-par/test_clean.ph < seq2seq.myph.hyp.txt

BLEU = 69.96, 86.1/74.1/65.2/57.7 (BP=1.000, ratio=1.000, hyp_len=8044, ref_len=8048)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.


### Grapheme-to-Phoneme Translation with Transformer Model
Now let try with transformer model

In [86]:
%%writefile transformer.myph.sh
#!/bin/bash

# ==============================================================================
# ENVIRONMENT PATHS
# ==============================================================================
BASE_DIR="/kaggle/working"
MODEL_FOLDER="${BASE_DIR}/model2.transformer.myph"
mkdir -p ${MODEL_FOLDER}

DATA_PATH="${BASE_DIR}/g2p-par" 
MARIAN_BIN="${BASE_DIR}/marian/build/marian"

src="my"
tgt="ph"

rm -f ${MODEL_FOLDER}/model.npz

# ==============================================================================
# MARIAN CONFIGURATION (HARDWARE-TUNED FOR TESLA T4 GPU)
# ==============================================================================
${MARIAN_BIN} \
    --type transformer \
    --train-sets ${DATA_PATH}/train_clean.${src} ${DATA_PATH}/train_clean.${tgt} \
    --max-length 200 \
    --valid-sets ${DATA_PATH}/dev_clean.${src} ${DATA_PATH}/dev_clean.${tgt} \
    --vocabs ${BASE_DIR}/vocab/vocab.${src}.yml ${BASE_DIR}/vocab/vocab.${tgt}.yml \
    --model ${MODEL_FOLDER}/model.npz \
    --mini-batch-fit -w 1000 --maxi-batch 100 \
    --early-stopping 10 \
    --valid-freq 2000 --save-freq 5000 --disp-freq 500 \
    --valid-metrics cross-entropy perplexity bleu \
    --valid-translation-output ${MODEL_FOLDER}/valid.${src}-${tgt}.output --quiet-translation \
    --valid-mini-batch 64 \
    --beam-size 6 --normalize 0.6 \
    --log ${MODEL_FOLDER}/train.log --valid-log ${MODEL_FOLDER}/valid.log \
    --enc-depth 2 --dec-depth 2 \
    --transformer-heads 8 \
    --transformer-postprocess-emb d \
    --transformer-postprocess dan \
    --transformer-dropout 0.3 --label-smoothing 0.1 \
    --learn-rate 0.0001 --lr-warmup 0 --lr-decay-inv-sqrt 16000 --lr-report \
    --clip-norm 1 \
    --tied-embeddings \
    --devices 0 --sync-sgd --seed 1111 \
    --exponential-smoothing \
    --dump-config > ${MODEL_FOLDER}/config.yml

echo "Launching high-throughput GPU training process..."
time ${MARIAN_BIN} -c ${MODEL_FOLDER}/config.yml 2>&1 | tee ${MODEL_FOLDER}/s2s.${src}-${tgt}.log

Overwriting transformer.myph.sh


In [87]:
!chmod +x transformer.myph.sh
!./transformer.myph.sh

Launching high-throughput GPU training process...
[2026-06-14 15:24:14] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-06-14 15:24:14] [marian] Running on f5f7ac2aebe0 as process 24398 with command line:
[2026-06-14 15:24:14] [marian] /kaggle/working/marian/build/marian -c /kaggle/working/model2.transformer.myph/config.yml
[2026-06-14 15:24:14] [config] after: 0e
[2026-06-14 15:24:14] [config] after-batches: 0
[2026-06-14 15:24:14] [config] after-epochs: 0
[2026-06-14 15:24:14] [config] all-caps-every: 0
[2026-06-14 15:24:14] [config] allow-unk: false
[2026-06-14 15:24:14] [config] authors: false
[2026-06-14 15:24:14] [config] beam-size: 6
[2026-06-14 15:24:14] [config] bert-class-symbol: "[CLS]"
[2026-06-14 15:24:14] [config] bert-mask-symbol: "[MASK]"
[2026-06-14 15:24:14] [config] bert-masking-fraction: 0.15
[2026-06-14 15:24:14] [config] bert-sep-symbol: "[SEP]"
[2026-06-14 15:24:14] [config] bert-train-type-embeddings: true
[2026-06-14 15:24:14] [config] bert-typ

In [88]:
!time marian/build/marian-decoder -m model2.transformer.myph/model.npz -v vocab/vocab.my.yml vocab/vocab.ph.yml --devices 0 < g2p-par/test_clean.my > transformer.myph.hyp.txt

[2026-06-14 15:40:13] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-06-14 15:40:13] [marian] Running on f5f7ac2aebe0 as process 24452 with command line:
[2026-06-14 15:40:13] [marian] marian/build/marian-decoder -m model2.transformer.myph/model.npz -v vocab/vocab.my.yml vocab/vocab.ph.yml --devices 0
[2026-06-14 15:40:13] [config] alignment: ""
[2026-06-14 15:40:13] [config] allow-special: false
[2026-06-14 15:40:13] [config] allow-unk: false
[2026-06-14 15:40:13] [config] authors: false
[2026-06-14 15:40:13] [config] beam-size: 12
[2026-06-14 15:40:13] [config] bert-class-symbol: "[CLS]"
[2026-06-14 15:40:13] [config] bert-mask-symbol: "[MASK]"
[2026-06-14 15:40:13] [config] bert-masking-fraction: 0.15
[2026-06-14 15:40:13] [config] bert-sep-symbol: "[SEP]"
[2026-06-14 15:40:13] [config] bert-train-type-embeddings: true
[2026-06-14 15:40:13] [config] bert-type-vocab-size: 2
[2026-06-14 15:40:13] [config] best-deep: false
[2026-06-14 15:40:13] [config] build-info: ""

In [89]:
!paste g2p-par/test_clean.my transformer.myph.hyp.txt | head -n 30 


တက် တက် ပြောင်	te' te' pjaun
ကပ် ပိ	ka' pi.
ရှုံ့ မဲ့	shoun. me.
ညှဉ်း ပန်း	njhin: ban:
မွမ်း မံ	mun: man
ငယ် မည်	nge mji
ရှာ ရှာ ဖွေ ဖွေ	sha sha hpwei hpwei
ပါး ပျဉ်း	pa- pjin:
ဥ မ ကွဲ သိုက် မ ပျက်	u. ma- gwe: thai' ma- pje'
အ စစ်	a- si'
ရေ ကျ	jei gya.
မ ဆုတ် မ ဆိုင်း	ma- hsou' ma- hsain:
ဟာ ကွက်	ha gwe'
မိန်း မူး	mein: mu:
ကယ် မ	ke ma.
နင်း နယ်	nin: ne
က ကြိုး တန် ဆာ	ga- gyou: tan za
ဆွမ်း ကြီး လောင်း	hswan: gyi: laun:
ဝါ ကြင့် ကြင့်	wa kyin. gyin.
ကုတ် ဟီး နာ	kou' hi: na
နီ ကြင် ကြင်	ni kyin gyin
အ လို တူ	a- lou du
ကိန်း ရင်း	kein: jin:
မီး တိုင်	mi: dain
ဝေ့ လည် ကြောင် ပတ်	wei. le kyaun ba'
စ ကား ပြော ကြေး နန်း	za- ga: pjo: gyei: nan:
အ ချစ် ဦး	a- chi' u:
အ သည်း ကောင်း	a- the: kaun:
ည ကြီး	nja. gyi:
ချို မြ မြ	gyou mja. mja.
paste: write error: Broken pipe
paste: write error


In [90]:
!perl mosesdecoder/scripts/generic/multi-bleu.perl g2p-par/test_clean.ph < transformer.myph.hyp.txt

BLEU = 71.87, 87.2/76.2/67.4/59.8 (BP=0.999, ratio=0.999, hyp_len=8041, ref_len=8048)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.
